# Prototype LLM API Caller

This notebook is a tiny starting point for calling an LLM API.
It has two simple examples:
1. Send one short text string and print the response.
2. Send a short company description and ask for a business summary plus commercial risks.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pypdf import PdfReader
from pathlib import Path
import json
import math

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

model_name = "gpt-5-mini"

client = OpenAI(api_key=api_key)

True


In [13]:
# Example 1: send one small text string to the model and print the answer.

sample_text = "Guidewire provides cloud software for property and casualty insurers."

response = client.responses.create(
    model=model_name,
    input=f"Read this short text and explain it in one simple sentence:\n\n{sample_text}"
)

print(response.output_text)

Guidewire makes cloud-based software used by companies that sell property and casualty insurance.


In [17]:
# Example 2: ask for a short business summary and a few possible commercial risks.

company_description = "Verisk sells data, analytics, and software products that help insurers price risk, manage claims, and support underwriting decisions."

prompt = f"""
You are helping with early-stage commercial due diligence.

Company description:
{company_description}

Please provide:
- a 2-bullet business summary
- 3 possible commercial risks

Keep the answer short and practical.
"""

response = client.responses.create(
    model=model_name,
    input=prompt
)

print(response.output_text)

- Business summary
  - Provides proprietary data, analytics and software to insurers and reinsurers to price risk, underwrite policies, and manage/settle claims (models, loss-costs, predictive analytics, workflow SaaS).
  - Enterprise, subscription-led model with high switching costs and regulatory/industry moats (longitudinal data, insurer relationships), but exposed to P&C market cycles and large-account selling dynamics.

- Three commercial risks
  1. Regulatory and data-privacy constraints: tighter data rules (privacy, AI explainability, classification of certain datasets) or limits on third‑party data sharing could reduce model inputs, slow product rollouts, or force costly reengineering—harming accuracy and differentiation.
  2. Technology/competitive disruption: cloud-native insurtechs, open-model AI providers, or hyperscaler analytics could offer lower‑cost or faster alternatives; loss of perceived predictive advantage would pressure pricing and churn among smaller customers.
 

In [ ]:
pdf_path = "Data/Guidewire/10-K.pdf"
reader = PdfReader(pdf_path)

text = ""
for page in reader.pages[:3]:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print(text[:3000])


In [ ]:
prompt = f"""
    You are a financial analyst reviewing a company's 10-K report. 
    Your task is to extract key information and insights from the document.
    The text from the first few pages is below:
    {text}
"""

response = client.responses.create(
    model=model_name,
    input=prompt
)

print(response.output_text)

## Breaking PDF text to chunks
Preparing for **embedding**

In [2]:
gw_path = Path("Data/Guidewire")

pdf_files = list(gw_path.glob("*.pdf"))

def extract_pdf_chunks(pdf_path, max_chars=3000, overlap=300):
    reader = PdfReader(pdf_path)
    chunks = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        i = 0

        while i < len(text):
            chunk = text[i:i + max_chars]

            chunks.append({
                "file_name": pdf_path.name,
                "page_number": page_num,
                "text": chunk
            })

            i += max_chars - overlap

    return chunks


all_chunks = []

for pdf_file in pdf_files:
    pages = extract_pdf_chunks(pdf_file)
    all_chunks.extend(pages)

print(len(all_chunks))

514


### Stupid version of `Retrieval` in `RAG`

In [6]:
def search_chunks(chunks, query, top_k=5):
    results = []
    query_words = query.lower().split()

    for chunk in chunks:
        text = chunk["text"].lower()
        score = 0

        for word in query_words:
            if word in text:
                score += 1

        if score > 0:
            results.append({
                "file_name": chunk["file_name"],
                "page_number": chunk["page_number"],
                "text": chunk["text"],
                "score": score
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
query = "Guidewire products InsuranceSuite InsuranceNow ClaimCenter BillingCenter platform applications"

results = search_chunks(all_chunks, query)

for result in results:
    print("score:", result["score"])
    print("file:", result["file_name"])
    print("page:", result["page_number"])
    print(result["text"][:300])
    print("-" * 80)

context = ""

for result in results:
    context += f"[{result['file_name']} - Page {result['page_number']}]\n"
    context += result['text'] + "\n\n"

score: 5
file: 10-Q.pdf
page: 31
access to our
innovation efforts. Additionally, InsuranceSuite embeds digital and analytics capabilities natively into our platform. Most new sales and
implementations are for InsuranceSuite.
InsuranceNow is a complete, cloud-based application that offers policy administration, claims management, an
--------------------------------------------------------------------------------
score: 5
file: 10-K.pdf
page: 7
Table of Contents
Item 1. Business
Overview and Purpose
Guidewire is the platform that property and casualty (“P&C”) insurers rely on to engage with customers, innovate, and operate
more efficiently. Founded in 2001, we serve insurers of all sizes, ranging from global carriers to regional and local 
--------------------------------------------------------------------------------
score: 5
file: 10-K.pdf
page: 15
Table of Contents
Guidewire Canvas is a cloud-native application included with ClaimCenter. It features an interactive map that enables cl

In [ ]:
prompt = f"""
    Use only the context below to answer the question.

    Question:
    {query}

    Context:
    {context}

    Please provide:
        - a short answer
        - 3 bullet summary
        - 3 possible commercial risks
"""

response = client.responses.create(
    model=model_name,
    input=prompt
)

print(response.output_text)


Short answer
Guidewire sells core insurance platform software and related cloud, digital, analytics and AI products — primarily InsuranceSuite and InsuranceNow plus a set of cloud-native apps and data/analytics offerings delivered on the Guidewire Cloud Platform (GWCP).

3‑bullet summary
- Core transactional systems: InsuranceSuite (PolicyCenter, ClaimCenter, BillingCenter, plus PricingCenter and UnderwritingCenter) and InsuranceNow (policy administration, claims, billing).
- Cloud/platform and services: Guidewire Cloud Platform (GWCP), cloud subscriptions, Guidewire Marketplace, support, professional services and term licenses.
- Digital/analytics/AI apps: digital engagement tools, Guidewire Canvas, Compare, Industry Intel, Data Studio & Explore, Cyence, DataHub/InfoCenter and other analytics/data products.

3 possible commercial risks (from the context)
- Long, complex sales and implementation cycles (especially for multi‑product deals or first‑time GWCP migrations) can slow bookings

## Embed all the chunks to vectors and store them as JSON format
`Please Note`: **This cell shouldn't be ran more than once for same chunks**

Later can be retrieved easily via: 


    with open("guidewire_chunks.json", "r") as f:
        saved_chunks = json.load(f)

    print(len(saved_chunks))
    print(saved_chunks[0].keys())


In [5]:
embedding_model = "text-embedding-3-small"

In [ ]:
for chunk in all_chunks:
    response = client.embeddings.create(
        model=embedding_model,
        input=chunk["text"]
    )
    chunk["embedding"] = response.data[0].embedding

print('done')
print(all_chunks[0].keys())

with open("guidewire_chunks.json", "w") as f:
    json.dump(all_chunks, f)

done
dict_keys(['file_name', 'page_number', 'text', 'embedding'])


In [6]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (norm_a * norm_b)

def search_chunks_by_embedding(chunks, query, top_k=5):
    query_response = client.embeddings.create(
        model=embedding_model,
        input=query
    )
    query_embedding = query_response.data[0].embedding

    results = []

    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        results.append({
            "file_name": chunk["file_name"],
            "page_number": chunk["page_number"],
            "text": chunk["text"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
query = "What products does Guidewire sell to insurance companies?"

with open("guidewire_chunks.json", "r") as f:
    saved_chunks = json.load(f)

results = search_chunks_by_embedding(saved_chunks, query)

for result in results:
    print("score:", result["score"])
    print("file:", result["file_name"])
    print("page:", result["page_number"])
    print(result["text"][:500])
    print("-" * 80)


In [8]:
saved_chunks[0].keys()

dict_keys(['file_name', 'page_number', 'text', 'embedding'])

In [13]:
results

[{'file_name': '10-K.pdf',
  'page_number': 11,
  'text': 'Table of Contents\n• Legacy System Modernization. A significant portion of the market continues to rely on legacy systems. We believe\nmodern policy administration, claims management, and billing systems will continue to be adopted as insurers that rely\non legacy systems seek to gain operating efficiencies, expand into new markets and lines of business, and introduce new\ndigital and data offerings.\nProducts\nThe Guidewire ecosystem is designed so that insurers can increase revenue, reduce operational costs and losses, improve pricing,\nand engage with a customer base that increasingly demands convenience and automated forms of self-service and communication. We\nare investing in research and development to accelerate improvements in our platform and suite of products to better serve our\ncustomers.\nCore Operational Products\nWe offer the following suite of products: Guidewire InsuranceSuite and Guidewire InsuranceNow.\nGuid

In [14]:
def answer_question(saved_chunks, query, top_k=5):
    results = search_chunks_by_embedding(saved_chunks, query, top_k=top_k)

    context = "\n\n".join(
        [
            f"File: {r['file_name']}, Page: {r['page_number']}\n{r['text']}"
            for r in results
        ]
    )

    prompt = f"""
        Use only the context below to answer the question.
        If the context is not sufficient, say what is missing.
        Include file and page references when making factual claims.

        Question:
        {query}

        Context:
        {context}

        Please provide:
        - a short answer
        - 3 bullet summary points
        - 3 possible commercial risks
    """

    response = client.responses.create(
        model=model_name,
        input=prompt
    )

    return {
        "query": query,
        "answer": response.output_text,
        "results": results,
    }


In [15]:
queries = [
    "What products does Guidewire sell to insurance companies?",
    "Who are Guidewire's target customers and buyer types?",
    "What are the main commercial risks for Guidewire?",
    "What evidence is there for Guidewire's competitive strengths or differentiation?",
    "What follow-up commercial diligence questions should an investor ask about Guidewire?",
]

for query in queries:
    qa_result = answer_question(saved_chunks, query, top_k=5)
    print(f"QUESTION: {qa_result['query']}")
    print()
    print(qa_result['answer'])
    print('\n' + '=' * 100 + '\n')


QUESTION: What products does Guidewire sell to insurance companies?

Short answer
Guidewire sells core cloud-delivered insurance systems — primarily Guidewire InsuranceSuite and Guidewire InsuranceNow (which include PolicyCenter, ClaimCenter and BillingCenter) — plus complementary digital, data/analytics, product design and platform services (e.g., Jutro Digital Engagement, Guidewire Predict, Reinsurance Management, Client Data Management, Advanced Product Designer, Product Content Management, Rating Management, Marketplace and the Guidewire Cloud Platform/GWCP). (10-K.pdf, pp. 7, 11; 10-K.pdf, p. 13; IR.pdf, p. 6)

3‑point summary
- Core operational products: InsuranceSuite (cloud service built on GWCP with PolicyCenter, ClaimCenter, BillingCenter) and InsuranceNow (complete cloud-based policy/claims/billing for mid‑market US carriers). (10-K.pdf, pp. 11, 7, 11)
- Digital and engagement offerings: Jutro-powered Digital Engagement Applications to deliver omnichannel, self‑service exper